In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
data = pd.read_csv("/content/drive/MyDrive/AML_Lab/credit_risk_assessment_500_samples.csv")

In [3]:
print("First 5 Rows:")
print(data.head())

First 5 Rows:
   Applicant_ID  Age  Annual_Income  Employment_Years  Credit_Score  \
0             1   59         153267                28           818   
1             2   49         232745                 0           436   
2             3   35         974945                19           797   
3             4   63         307164                29           776   
4             5   28         685626                 6           893   

   Loan_Amount  Loan_Term_Months  Existing_Loans_Count  Debt_to_Income_Ratio  \
0       196649                12                     2                  0.68   
1       175354                60                     0                  0.23   
2       662297                48                     3                  0.54   
3       831725                48                     0                  0.56   
4       498625                60                     3                  0.66   

   Late_Payments_Last_2Yrs Credit_Risk  
0                        0        Hig

In [4]:
print("\nDataset Info:")
print(data.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Applicant_ID             500 non-null    int64  
 1   Age                      500 non-null    int64  
 2   Annual_Income            500 non-null    int64  
 3   Employment_Years         500 non-null    int64  
 4   Credit_Score             500 non-null    int64  
 5   Loan_Amount              500 non-null    int64  
 6   Loan_Term_Months         500 non-null    int64  
 7   Existing_Loans_Count     500 non-null    int64  
 8   Debt_to_Income_Ratio     500 non-null    float64
 9   Late_Payments_Last_2Yrs  500 non-null    int64  
 10  Credit_Risk              500 non-null    object 
dtypes: float64(1), int64(9), object(1)
memory usage: 43.1+ KB
None


In [5]:
data = data.drop_duplicates()

In [6]:
data = data.fillna(data.median(numeric_only=True))

In [7]:
X = data.drop("Credit_Risk", axis=1)
y = data["Credit_Risk"]

In [8]:
X = pd.get_dummies(X, drop_first=True)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [12]:
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train_encoded)

LogisticRegression()

In [13]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train_encoded)

RandomForestClassifier(random_state=42)

In [14]:
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:59:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
log_pred = log_model.predict(X_test_scaled)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

In [16]:
log_prob = log_model.predict_proba(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test)
xgb_prob = xgb_model.predict_proba(X_test)

In [17]:
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n{name} Performance:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall:", recall_score(y_true, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

In [18]:
evaluate_model("Logistic Regression", y_test_encoded, log_pred, log_prob)
evaluate_model("Random Forest", y_test_encoded, rf_pred, rf_prob)
evaluate_model("XGBoost", y_test_encoded, xgb_pred, xgb_prob)


Logistic Regression Performance:
Accuracy: 0.84
Precision: 0.8373701298701297
Recall: 0.84
F1 Score: 0.8364478551751279

Random Forest Performance:
Accuracy: 0.96
Precision: 0.9617079530638852
Recall: 0.96
F1 Score: 0.9541404794036373

XGBoost Performance:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


In [19]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/AML_LAB/"

pickle.dump(log_model, open(save_path + "logistic_model.pkl", "wb"))
pickle.dump(rf_model, open(save_path + "random_forest_model.pkl", "wb"))
pickle.dump(xgb_model, open(save_path + "xgboost_model.pkl", "wb"))

print("\nModels saved successfully in Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Models saved successfully in Google Drive!


In [20]:
def print_stats(name, y_true, y_pred):
    print(f"\n--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")

print_stats("Logistic Regression", y_test_encoded, log_model.predict(X_test_scaled))
print_stats("Random Forest", y_test_encoded, rf_model.predict(X_test))
print_stats("XGBoost", y_test_encoded, xgb_model.predict(X_test))


--- Logistic Regression ---
Accuracy: 0.8400
Precision: 0.8374

--- Random Forest ---
Accuracy: 0.9600
Precision: 0.9617

--- XGBoost ---
Accuracy: 1.0000
Precision: 1.0000


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import auc

classes = label_encoder.classes_
y_test_bin = label_binarize(y_test_encoded, classes=range(len(classes)))
n_classes = len(classes)

# Define models and their probabilities for the loop
models_prob = {
    "Logistic Regression": log_prob,
    "Random Forest": rf_prob,
    "XGBoost": xgb_prob
}

# Create a figure with 3 subplots side-by-side
plt.figure(figsize=(18, 6))

for i in range(n_classes):
    plt.subplot(1, n_classes, i + 1)

    for model_name, prob in models_prob.items():
        # Calculate ROC curve and Area Under Curve (AUC) for the specific class
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], prob[:, i])
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

    # Draw the diagonal random guess line (0.5 AUC)
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')

    # Graph Styling
    plt.title(f'ROC Curve for Class: {classes[i]}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right', fontsize='small')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()